In [ ]:
import requests
import pandas as pd 

# Get all F1 drivers in history
response = requests.get("http://api.jolpi.ca/ergast/f1/drivers.json?limit=1000")
data = response.json()

print(response.status_code)
print(data['MRData']['total'])  # Shows total number of drivers ever

200
879


In [ ]:
all_drivers = []

for year in range(2005, 2026):
    response = requests.get(f"http://api.jolpi.ca/ergast/f1/{year}/drivers.json?limit=100")
    data = response.json()
    drivers = data['MRData']['DriverTable']['Drivers']
    
    for driver in drivers:
        driver['season'] = year  # Fetch all drivers who raced between 2005-2025 
    
    all_drivers.extend(drivers)

df = pd.DataFrame(all_drivers)
print(df.shape)
df.head(514)

(514, 9)


,driverId,code,url,givenName,familyName,dateOfBirth,nationality,season,permanentNumber
0,albers,ALB,http://en.wikipedia.org/wiki/Christijan_Albers,Christijan,Albers,1979-04-16,Dutch,2005,NaN
1,alonso,ALO,http://en.wikipedia.org/wiki/Fernando_Alonso,Fernando,Alonso,1981-07-29,Spanish,2005,14
2,barrichello,BAR,http://en.wikipedia.org/wiki/Rubens_Barrichello,Rubens,Barrichello,1972-05-23,Brazilian,2005,NaN
3,button,BUT,http://en.wikipedia.org/wiki/Jenson_Button,Jenson,Button,1980-01-19,British,2005,22
4,coulthard,COU,http://en.wikipedia.org/wiki/David_Coulthard,David,Coulthard,1971-03-27,British,2005,NaN
...,...,...,...,...,...,...,...,...,...
509,cian_shields,NaN,NaN,Cian,Shields,NaN,NaN,2025,NaN
510,stroll,STR,http://en.wikipedia.org/wiki/Lance_Stroll,Lance,Stroll,1998-10-29,Canadian,2025,18
511,tsunoda,TSU,http://en.wikipedia.org/wiki/Yuki_Tsunoda,Yuki,Tsunoda,2000-05-11,Japanese,2025,22
512,max_verstappen,VER,http://en.wikipedia.org/wiki/Max_Verstappen,Max,Verstappen,1997-09-30,Dutch,2025,3


In [ ]:
all_results = []

for year in range(2005, 2026):
    response = requests.get(f"http://api.jolpi.ca/ergast/f1/{year}/results.json?limit=1000")
    data = response.json()
    races = data['MRData']['RaceTable']['Races']
    
    for race in races:
        round_num = race['round']
        race_name = race['raceName']
        for result in race['Results']:
            result['season'] = year
            result['round'] = round_num
            result['raceName'] = race_name
            all_results.append(result) # Fetch all race results (2005-2025)

df_results = pd.DataFrame(all_results)
print(df_results.shape)
df_results.head()

(2100, 14)


,number,position,positionText,points,Driver,Constructor,grid,laps,status,Time,FastestLap,season,round,raceName
0,6,1,1,10,"{'driverId': 'fisichella', 'code': 'FIS', 'url...","{'constructorId': 'renault', 'url': 'https://e...",1,57,Finished,"{'millis': '5057336', 'time': '1:24:17.336'}","{'rank': '2', 'lap': '55', 'Time': {'time': '1...",2005,1,Australian Grand Prix
1,2,2,2,8,"{'driverId': 'barrichello', 'code': 'BAR', 'ur...","{'constructorId': 'ferrari', 'url': 'https://e...",11,57,Finished,"{'millis': '5062889', 'time': '+5.553'}","{'rank': '3', 'lap': '54', 'Time': {'time': '1...",2005,1,Australian Grand Prix
2,5,3,3,6,"{'driverId': 'alonso', 'permanentNumber': '14'...","{'constructorId': 'renault', 'url': 'https://e...",13,57,Finished,"{'millis': '5064048', 'time': '+6.712'}","{'rank': '1', 'lap': '24', 'Time': {'time': '1...",2005,1,Australian Grand Prix
3,14,4,4,5,"{'driverId': 'coulthard', 'code': 'COU', 'url'...","{'constructorId': 'red_bull', 'url': 'https://...",5,57,Finished,"{'millis': '5073467', 'time': '+16.131'}","{'rank': '11', 'lap': '40', 'Time': {'time': '...",2005,1,Australian Grand Prix
4,7,5,5,4,"{'driverId': 'webber', 'code': 'WEB', 'url': '...","{'constructorId': 'williams', 'url': 'https://...",3,57,Finished,"{'millis': '5074244', 'time': '+16.908'}","{'rank': '8', 'lap': '37', 'Time': {'time': '1...",2005,1,Australian Grand Prix


In [5]:
# Fetch end-of-season driver championship standings (2005-2025)
# Each row = one driver's final championship position for that season
# Used to identify WDC winners (position == 1) for filtering later
all_standings = []

for year in range(2005, 2026):
    response = requests.get(f"http://api.jolpi.ca/ergast/f1/{year}/driverStandings.json?limit=100")
    data = response.json()
    standings_list = data['MRData']['StandingsTable']['StandingsLists']
    
    if standings_list:
        for entry in standings_list[0]['DriverStandings']:
            entry['season'] = year
            all_standings.append(entry)

df_standings = pd.DataFrame(all_standings)
print(df_standings.shape)
df_standings.head()

(498, 7)


,position,positionText,points,wins,Driver,Constructors,season
0,1,1,133,7,"{'driverId': 'alonso', 'permanentNumber': '14'...","[{'constructorId': 'renault', 'url': 'https://...",2005
1,2,2,112,7,"{'driverId': 'raikkonen', 'permanentNumber': '...","[{'constructorId': 'mclaren', 'url': 'https://...",2005
2,3,3,62,1,"{'driverId': 'michael_schumacher', 'code': 'MS...","[{'constructorId': 'ferrari', 'url': 'https://...",2005
3,4,4,60,3,"{'driverId': 'montoya', 'code': 'MON', 'url': ...","[{'constructorId': 'mclaren', 'url': 'https://...",2005
4,5,5,58,1,"{'driverId': 'fisichella', 'code': 'FIS', 'url...","[{'constructorId': 'renault', 'url': 'https://...",2005


In [6]:
# Fetch end-of-season constructor championship standings (2005-2025)
# Each row = one constructor's final championship position for that season
# Used to assess how competitive a driver's car was in their peak years
all_constructor_standings = []

for year in range(2005, 2026):
    response = requests.get(f"http://api.jolpi.ca/ergast/f1/{year}/constructorStandings.json?limit=100")
    data = response.json()
    standings_list = data['MRData']['StandingsTable']['StandingsLists']
    
    if standings_list:
        for entry in standings_list[0]['ConstructorStandings']:
            entry['season'] = year
            all_constructor_standings.append(entry)

df_constructor_standings = pd.DataFrame(all_constructor_standings)
print(df_constructor_standings.shape)
df_constructor_standings.head()

(223, 6)


,position,positionText,points,wins,Constructor,season
0,1,1,191,8,"{'constructorId': 'renault', 'url': 'https://e...",2005
1,2,2,182,10,"{'constructorId': 'mclaren', 'url': 'https://e...",2005
2,3,3,100,1,"{'constructorId': 'ferrari', 'url': 'https://e...",2005
3,4,4,88,0,"{'constructorId': 'toyota', 'url': 'https://en...",2005
4,5,5,66,0,"{'constructorId': 'williams', 'url': 'https://...",2005


In [7]:
# Identify WDC winners — drivers who finished P1 in championship standings
# Extract driverId from nested Driver dict first
df_standings['driverId'] = df_standings['Driver'].apply(lambda x: x['driverId'])

wdc_winners = df_standings[df_standings['position'] == '1']['driverId'].unique()
print(f"WDC winners in 2005-2025: {wdc_winners}")
print(f"Total: {len(wdc_winners)}")

WDC winners in 2005-2025: <StringArray>
[        'alonso',      'raikkonen',       'hamilton',         'button',
         'vettel',        'rosberg', 'max_verstappen',         'norris']
Length: 8, dtype: str
Total: 8


In [8]:
# Extract driverId from results Driver dict
df_results['driverId'] = df_results['Driver'].apply(lambda x: x['driverId'])

# Filter to non-champions only
non_champions = df_results[~df_results['driverId'].isin(wdc_winners)]

print(f"Total race entries: {len(df_results)}")
print(f"Non-champion race entries: {len(non_champions)}")
print(f"Unique non-champion drivers: {non_champions['driverId'].nunique()}")

Total race entries: 2100
Non-champion race entries: 1568
Unique non-champion drivers: 85


In [12]:
# Re-create non_champions now that constructorId exists on df_results
non_champions = df_results[~df_results['driverId'].isin(wdc_winners)]

# Now merge
merged = non_champions.merge(
    df_constructor_standings[['constructorId', 'season', 'position']].rename(columns={'position': 'constructor_rank'}),
    on=['constructorId', 'season'],
    how='left'
)

print(merged.shape)
merged[['driverId', 'constructorId', 'season', 'position', 'constructor_rank', 'status']].head()

(1568, 17)


,driverId,constructorId,season,position,constructor_rank,status
0,fisichella,renault,2005,1,1,Finished
1,barrichello,ferrari,2005,2,3,Finished
2,coulthard,red_bull,2005,4,7,Finished
3,webber,williams,2005,5,5,Finished
4,montoya,mclaren,2005,6,2,Finished


In [15]:
# Clean merged dataset - fix dtypes and handle DNFs
# Convert key columns to numeric
merged['position'] = pd.to_numeric(merged['position'], errors='coerce')
merged['constructor_rank'] = pd.to_numeric(merged['constructor_rank'], errors='coerce')
merged['grid'] = pd.to_numeric(merged['grid'], errors='coerce')
merged['laps'] = pd.to_numeric(merged['laps'], errors='coerce')
merged['points'] = pd.to_numeric(merged['points'], errors='coerce')

# Flag DNFs - lapped cars ('+1 Lap' etc.) still finished, so exclude them
finished_statuses = ['Finished', '+1 Lap', '+2 Laps', '+3 Laps', '+4 Laps', '+5 Laps',
                     '+6 Laps', '+7 Laps', '+8 Laps', '+9 Laps', 'Lapped']
merged['dnf'] = ~merged['status'].isin(finished_statuses)

# Check results
print(merged.dtypes)
print(f"\nDNF count: {merged['dnf'].sum()}")
print(f"DNF rate: {merged['dnf'].mean():.1%}")

number                  str
position              int64
positionText            str
points              float64
Driver               object
Constructor          object
grid                  int64
laps                  int64
status                  str
Time                 object
FastestLap           object
season                int64
round                   str
raceName                str
driverId                str
constructorId           str
constructor_rank      int64
dnf                    bool
dtype: object

DNF count: 355
DNF rate: 22.6%


In [14]:
# See all unique status values
merged['status'].value_counts().head(20)

status
Finished        702
+1 Lap          348
+2 Laps          81
Collision        63
Lapped           49
Retired          48
Accident         29
Engine           27
Hydraulics       23
Gearbox          18
+3 Laps          17
Spun off         17
Brakes           11
Suspension       10
Electrical       10
+4 Laps           9
Power Unit        9
Disqualified      8
Clutch            6
Wheel             6
Name: count, dtype: int64